# Fixed-start benchmark evidence and portable smoke run

This notebook validates the released summary grid, preserves estimator failures, and executes one clean reference condition from the frozen production implementation. It does not repeat the full benchmark or reinterpret it as biological generalization. The adjacent result notebooks contain the existing scientific plots; this audit uses tables.

The current execution uses the neutral local environment alias and preserves the historical numerical inputs, cores, numerical module, and aggregate hash contract.

In [1]:
from pathlib import Path
import hashlib, importlib.metadata, json, platform, sys
import numpy as np
import pandas as pd
from IPython.display import display

def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda: handle.read(1048576), b''):
            digest.update(block)
    return digest.hexdigest()

HISTORICAL_MAC_ENVIRONMENT = {
    'manager': 'existing micromamba-compatible environment, reused without modification',
    'name': 'chronoclone',
    'absolute_macbook_path': '/path/to/ChronoClone/.runtime/chronoclone',
    'kernel': 'chronoclone-paper',
}
runtime = dict(HISTORICAL_MAC_ENVIRONMENT,
    current_prefix=sys.prefix, current_executable=sys.executable, python=platform.python_version(),
    versions={name: importlib.metadata.version(name) for name in
              ['numpy','pandas','scipy','scikit-learn','scanpy','matplotlib','nbformat','nbclient']})

MODULE = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
              if (p / 'results/aggregate/aggregation_metadata.json').is_file())
BENCHMARKS = MODULE.parent
REPORTS = MODULE / 'reports'
REPORTS.mkdir(exist_ok=True)
runtime['execution_command'] = 'python benchmarks/run_notebook.py benchmarks/ode_noise/notebooks/0006_benchmark_evidence_and_smoke.ipynb --kernel chronoclone-paper'
display(pd.DataFrame([runtime]).T)


,0
manager,"existing micromamba-compatible environment, re..."
name,chronoclone
absolute_macbook_path,/path/to/ChronoClone/.runt...
kernel,chronoclone-paper
current_prefix,/path/to/ChronoClone/.runt...
current_executable,/path/to/ChronoClone/.runt...
python,3.11.15
versions,"{'numpy': '2.4.6', 'pandas': '2.3.3', 'scipy':..."
execution_command,python benchmarks/run_notebook.py benchmarks/o...


## Verify the released identities

Hashes identify the stored evidence. They do not by themselves establish that every historical fit is correct.

In [2]:
AGGREGATE = MODULE / 'results/aggregate'
metadata = json.loads((AGGREGATE / 'aggregation_metadata.json').read_text())
assert metadata['status'] == 'pass'
records = []
for name, expected in metadata['artifact_sha256'].items():
    path = AGGREGATE / name
    observed = sha256(path)
    assert observed == expected, name
    records.append({'artifact': name, 'bytes': path.stat().st_size, 'sha256_matches': observed == expected})
display(pd.DataFrame(records))
large = metadata['omitted_large_artifacts']['native_method_noise_runs.csv.gz']
full_path = AGGREGATE / 'native_method_noise_runs.csv.gz'
if full_path.is_file():
    assert full_path.stat().st_size == large['bytes'] and sha256(full_path) == large['sha256']
print('Repeat-level aggregate available locally:', full_path.is_file())
print('Repeat-level asset requirement:', json.dumps(large, indent=2))


,artifact,bytes,sha256_matches
0,chronoclone_selective_summary.csv,2366462,True
1,chunk_manifest.csv,125307,True
2,eligibility_summary.csv,1376685,True
3,method_failure_counts.csv,320802,True
4,native_method_noise_summary.csv,6226263,True
5,result_completeness.csv,1233999,True
6,same_retained_subset_summary.csv,6977272,True


Repeat-level aggregate available locally: True
Repeat-level asset requirement: {
  "bytes": 153477867,
  "reason": "Exceeds GitHub's 100 MB per-file limit; regenerate from the supplied runner or retrieve from the versioned archive when deposited.",
  "sha256": "e206600a20219a4b8bf11f8213e13ad0909c8ade96c31b33b9c45451d06a984a"
}


## Check grid completeness and estimator semantics

Each coordinate has its own eligible clone set. The five-method aggregate is retained; the main comparison displays the four estimator-compatible methods. A failed estimator has no ARI, rather than zero ARI.

In [3]:
from itertools import product
summary = pd.read_csv(AGGREGATE / 'native_method_noise_summary.csv')
complete = pd.read_csv(AGGREGATE / 'result_completeness.csv')
selective = pd.read_csv(AGGREGATE / 'chronoclone_selective_summary.csv')
eligibility = pd.read_csv(AGGREGATE / 'eligibility_summary.csv')
same_retained = pd.read_csv(AGGREGATE / 'same_retained_subset_summary.csv')
methods = ['ChronoClone','CLADES preprocessing','Cyclone abundance-only','K-means','Ward']
levels = list(range(0,101,2))
keys = ['method','gaussian_pct','uniform_pct']
expected = set(product(methods,levels,levels))
for frame in [summary,complete,same_retained]:
    assert not frame.duplicated(keys).any()
    assert set(frame[keys].itertuples(index=False,name=None)) == expected
    assert frame.n_expected.eq(100).all()
    assert (frame.n_success + frame.n_method_failure).eq(100).all()
for frame in [selective,eligibility]:
    assert not frame.duplicated(['gaussian_pct','uniform_pct']).any()
    assert set(frame[['gaussian_pct','uniform_pct']].itertuples(index=False,name=None)) == set(product(levels,levels))
assert int(complete.n_expected.sum()) == metadata['observed_rows']
assert int(complete.n_method_failure.sum()) == metadata['method_failure_rows']
joined = summary[keys+['n_success','n_method_failure']].merge(complete[keys+['n_success','n_method_failure']],on=keys,validate='one_to_one',suffixes=('_summary','_complete'))
assert joined.n_success_summary.equals(joined.n_success_complete)
assert joined.n_method_failure_summary.equals(joined.n_method_failure_complete)
assert summary.loc[summary.n_success.eq(0), 'ari_mean'].isna().all()
assert summary.ari_n_finite.equals(summary.n_success)
assert summary.loc[summary.n_success.gt(0),'ari_mean'].between(-1,1).all()
assert summary.loc[summary.n_success.gt(0),'ari_q025'].le(summary.loc[summary.n_success.gt(0),'ari_q975']).all()
primary = complete.loc[complete.method.ne('Cyclone abundance-only')]
assert primary.n_success.eq(100).all()
assert selective.postfit_retained_fraction_of_eligible_mean.between(0,1).all()
assert eligibility.eligible_fraction_mean.between(0,1).all()
display(complete.groupby('method',sort=False)[['n_expected','n_success','n_method_failure']].sum())
display(summary.loc[summary.gaussian_pct.eq(0)&summary.uniform_pct.eq(0),['method','n_success','ari_mean','recovered_k_mean']])


,n_expected,n_success,n_method_failure
method,,,
ChronoClone,260100,260100,0
CLADES preprocessing,260100,260100,0
Cyclone abundance-only,260100,11934,248166
K-means,260100,260100,0
Ward,260100,260100,0


,method,n_success,ari_mean,recovered_k_mean
0,ChronoClone,100,1.0,12.0
2601,CLADES preprocessing,100,1.0,12.0
5202,Cyclone abundance-only,0,NaN,NaN
7803,K-means,100,1.0,12.0
10404,Ward,100,1.0,12.0


## Execute one bounded condition

The wrapper verifies the frozen configuration, corrected core, low-level core, reference, and original numerical module. It preserves source bytes and loads them through temporary paths. It requires the recorded dependency versions and writes a separate smoke record. The four fits finish before truth and retention metrics are evaluated. This is a clean-reference portability test, not a full-grid rerun.

In [4]:
import subprocess
smoke_output = REPORTS / 'portable_clean_smoke.json'
command = [sys.executable, str(MODULE/'scripts/run_smoke.py'), '--output',str(smoke_output)]
completed = subprocess.run(command,capture_output=True,text=True)
if completed.returncode:
    raise RuntimeError(completed.stdout + '\n' + completed.stderr)
smoke = json.loads(smoke_output.read_text())
assert smoke['status']=='pass' and smoke['t1_changed_entries']==0
assert smoke['runtime']['historical_versions_match']
display(pd.DataFrame(smoke['results']))
display(pd.DataFrame([{k:smoke[k] for k in ['eligible_clones','postfit_retained_clones','postfit_retention_fraction','t1_changed_entries']}]))
if completed.stderr.strip():
    print('Runtime information captured:', completed.stderr.strip()[:1500])


,method,ari,nmi,recovered_k,selected_parameter,label_sha256
0,ChronoClone,1.0,1.0,12,0.01,fae57691817e584c81ab8f68885baa337ae4a59e8cc84f...
1,CLADES preprocessing,1.0,1.0,12,0.25,bc38bc45b028b882109acdac2f58f7796a5e20e83007c6...
2,K-means,1.0,1.0,12,12.00,955a013168247c889e00b67454b63984c4181d70fa7c6d...
3,Ward,1.0,1.0,12,12.00,02f339593eb5ae50f8653564ae69904d37a57d05e613dc...


,eligible_clones,postfit_retained_clones,postfit_retention_fraction,t1_changed_entries
0,517,517,1.0,0


Runtime information captured: Matplotlib is building the font cache; this may take a moment.


## Record the evidence boundary

The compact tables and bounded run are validated separately. Full-grid regeneration still uses the documented historical production topology; the new wrapper does not resume its shards. Monte Carlo ranges describe this generator. They are not biological confidence intervals.

In [5]:
result = {'schema':'chronoclone_paper_benchmark_evidence_v1','status':'pass',
          'summary_rows':len(summary),'grid_coordinates':len(eligibility),
          'historical_method_condition_rows':int(complete.n_expected.sum()),
          'historical_method_failures':int(complete.n_method_failure.sum()),
          'summary_hashes_verified':len(records),'bounded_smoke_status':smoke['status'],
          'full_aggregate_verified_here':full_path.is_file(),
          'full_benchmark_rerun':False,'runtime':runtime}
(REPORTS/'benchmark_evidence_validation.json').write_text(json.dumps(result,indent=2)+'\n')
display(pd.DataFrame([{key:value for key,value in result.items() if key!='runtime'}]).T)


,0
schema,chronoclone_paper_benchmark_evidence_v1
status,pass
summary_rows,13005
grid_coordinates,2601
historical_method_condition_rows,1300500
historical_method_failures,248166
summary_hashes_verified,7
bounded_smoke_status,pass
full_aggregate_verified_here,True
full_benchmark_rerun,False
